# Reporte del dataset

Inventario de lo que hay en `data/cleaned/` y en el caché de `data/processed/`: archivos que
entran, anotaciones totales, reparto por clase y por split, y una huella (`sha256`) de la lista
de grabaciones de cada split para comparar dos copias del dataset de un vistazo.

Ejecutar desde la raíz del repo o desde `notebooks/`.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from core.config import settings  # noqa: E402
from data import cache  # noqa: E402
from data.annotations import load_annotations  # noqa: E402
from prepare_data import select_experiment  # noqa: E402

pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)
print(ROOT)

/home/fcandia/tesis-primate


## 1. `data/cleaned/` — anotaciones normalizadas

Una `.txt` por grabación con `.wav` al lado en `data/raw/`; es lo que lee todo lo demás.

In [2]:
annotations = load_annotations()
annotations["recording"] = [
    str(Path(p).relative_to(settings.raw_dir)) for p in annotations["audio_path"]
]
annotations["folder"] = annotations["recording"].str.split("/").str[0]

cleaned_files = sorted(p.relative_to(settings.cleaned_dir) for p in settings.cleaned_dir.rglob("*.txt"))
print(f"archivos .txt en cleaned/ : {len(cleaned_files)}")
print(f"grabaciones con .wav      : {annotations['recording'].nunique()}")
print(f"anotaciones totales       : {len(annotations)}")
print(f"especies                  : {annotations['species'].nunique()}")
print(f"pares species/call_type   : {annotations.groupby(['species', 'call_type']).ngroups}")

archivos .txt en cleaned/ : 2819
grabaciones con .wav      : 2792
anotaciones totales       : 19033
especies                  : 8
pares species/call_type   : 65


In [3]:
por_carpeta = (
    annotations.groupby("folder")
    .agg(
        grabaciones=("recording", "nunique"),
        anotaciones=("recording", "size"),
        tipos_llamada=("call_type", "nunique"),
        dur_media_s=("duration_s", "mean"),
        bw_medio_hz=("bandwidth_hz", "mean"),
    )
    .round(2)
)
por_carpeta.loc["TOTAL"] = [
    annotations["recording"].nunique(),
    len(annotations),
    annotations.groupby(["species", "call_type"]).ngroups,
    round(annotations["duration_s"].mean(), 2),
    round(annotations["bandwidth_hz"].mean(), 2),
]
por_carpeta

,grabaciones,anotaciones,tipos_llamada,dur_media_s,bw_medio_hz
folder,,,,,
bolivian_squirrel_monkey__SB,256.0,2337.0,9.0,0.312612,8290.796299
howler_monkey__AS,773.0,3566.0,8.0,5.316782,672.156855
large_headed_capuchin__SM,571.0,3621.0,12.0,0.271921,2969.644291
night_monkey__AA,82.0,1094.0,5.0,0.137757,4078.197625
peruvian_spider_monkey__AC,379.0,3085.0,6.0,0.590762,2922.898466
shock_headed_capuchin_monkey__CC,7.0,29.0,1.0,0.141733,3844.175724
toppins_titi_monkey__PT,315.0,1157.0,7.0,4.717851,1647.094537
weddells_saddleBack_tamarin__LW,409.0,4144.0,17.0,0.683347,3352.168575
TOTAL,2792.0,19033.0,65.0,1.63,3252.92


In [4]:
# Todos los pares species/call_type de cleaned/, incluidos los que el experimento descarta.
pares_cleaned = (
    annotations.groupby(["species", "call_type"])
    .agg(anotaciones=("recording", "size"), grabaciones=("recording", "nunique"))
    .sort_values("anotaciones", ascending=False)
)
print(f"{len(pares_cleaned)} pares | {int(pares_cleaned['anotaciones'].sum())} anotaciones")
pares_cleaned

65 pares | 19033 anotaciones


anotaciones  grabaciones
species call_type                          
ac      bc                2487          138
lw      cs                2282          248
sm      cc                2040          467
as      bc                1909          237
        hc                1400          706
sb      ppc               1151          188
aa      gc                 799           58
sb      spc                729          166
sm      fs                 691           89
pt      dc                 637          294
lw      cc                 536          250
pt      sqc                392           13
lw      tr                 332           89
sb      pcc                290           95
lw      ta                 265           80
sm      hic                254           61
        pc                 242           83
ac      chc                236          143
sm      sc                 168           33
lw      tj                 164           34
aa      sc                 159           33
ac      sc                 158           31
lw      sqc                154           43
ac      gc                 140           65
aa      hm                 129           24
sm      fc                 126           91
lw      vc                 119           44
        tt                 106           31
sb      sc                 105           51
as      cp                  91           31
        pp                  85           31
lw      aa                  80           37
sm      whc                 76           22
as      ip                  73           29
sb      lpc                 49           19
pt      ac                  44           17
        pp                  43            8
        bp                  39            9
ac      whc                 35           29
lw      phc                 34           15
        tf                  31           17
cc      cc                  29            7
ac      cc                  29           13
lw      a                   20            6
        b                   15            8
sm      acc                  8            7
sb      pcs                  8            2
sm      hc                   8            3
aa      sqc                  6            1
sm      sic                  4            1
as      cc                   4            1
sm      chc                  3            2
sb      phc                  3            1
as      pc                   3            1
lw      c                    3            3
aa      hf                   1            1
as      chc                  1            1
lw      t                    1            1
        lw                   1            1
        chc                  1            1
pt      d                    1            1
sb      hic                  1            1
        cc                   1            1
pt      chc                  1            1
sm      spc                  1            1

## 2. El experimento: qué pares sobreviven al filtro

`select_experiment()` es la misma función que usa `prepare_data.py`: aplica `EXCLUDED_PAIRS`,
`JOINED_PAIRS` y `MIN_PAIR_COUNT`. Si esto no coincide entre dos copias, el caché tampoco.

In [5]:
experiment_df, labels = select_experiment()
experiment_df["recording"] = [
    str(Path(p).relative_to(settings.raw_dir)) for p in experiment_df["audio_path"]
]

print(f"anotaciones del experimento : {len(experiment_df)}")
print(f"grabaciones                 : {experiment_df['recording'].nunique()}")
print(f"clases                      : {len(labels)}")
print(", ".join(labels.names))

anotaciones del experimento : 17569
grabaciones                 : 2668
clases                      : 25
aa/gc, aa/hm, aa/sc, ac/bc, ac/chc, ac/gc, ac/sc, as/bc, as/hc, lw/cs, lw/sqc, lw/ta, lw/trino, lw/vc, pt/dc, pt/sqc, sb/pcc, sb/ppc, sb/sc, sb/spc, sm/cc, sm/fs, sm/hic, sm/pc, sm/sc


## 3. `data/processed/` — metadatos del caché

In [6]:
meta = cache.meta()
labels_json = cache.read_json("labels.json")
print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
print(f"{len(labels_json)} clases en labels.json")

{
  "seed": 42,
  "min_pair_count": 100,
  "max_classes": null,
  "empty_ratio": 0.25,
  "label_by": "species/call_type",
  "excluded_pairs": [
    [
      "lw",
      "cc"
    ],
    [
      "sb",
      "pcs"
    ],
    [
      "sm",
      "fc"
    ]
  ],
  "normalization": {
    "mean": 4.385143528598441,
    "std": 212.34048583374556
  },
  "db_range": [
    -54.41232303619385,
    28.752450983047567
  ],
  "params": {
    "clip_len_s": 3.0,
    "clip_hop_s": 1.5,
    "min_overlap": 0.5,
    "pad_seed": 0,
    "target_sr": 44100,
    "n_fft": 4096,
    "win_length": 1024,
    "hop_length": 400,
    "n_mels": 128,
    "f_min": 25.0,
    "f_max": 22050.0,
    "mel_scale": "htk",
    "mel_break_hz": 700.0,
    "mel_scale_q": 2595.0,
    "eps": 1e-06
  }
}

25 clases en labels.json


In [7]:
archivos_cache = sorted(settings.processed_dir.iterdir())
pd.DataFrame(
    [
        {
            "archivo": p.name,
            "MB": round(p.stat().st_size / 1024**2, 2),
            "sha256_16": hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.stat().st_size < 5e7 else "",
        }
        for p in archivos_cache
    ]
)

,archivo,MB,sha256_16
0,labels.json,0.00,23324ef53740994d
1,meta.json,0.00,9a8e9168e7347f5a
2,test.pt,1292.17,
3,test_sources.json,0.19,9965134cc24537fc
4,train.pt,3547.42,
5,train_sources.json,0.51,fe5c27e6634c134c
6,val.pt,1543.11,
7,val_sources.json,0.23,9a92292ccec4c795


## 4. Splits: ventanas, grabaciones y cajas

`*.pt` se abre con `mmap=True`: se leen las cajas y las etiquetas sin traer los mel a memoria
(los tres juntos pesan más de 6 GB).

In [8]:
def read_split(name: str) -> dict:
    return torch.load(cache.split_path(name), map_location="cpu", mmap=True, weights_only=True)


splits = {name: read_split(name) for name in cache.SPLITS}
sources = {
    name: cache.sources(name, len(data["images"])) for name, data in splits.items()
}
recordings = {
    name: sorted(str(Path(p).relative_to(settings.raw_dir)) for p in src.recordings)
    for name, src in sources.items()
}
{name: len(rs) for name, rs in recordings.items()}

{'train': 1324, 'val': 724, 'test': 619}

In [9]:
filas = []
for name, data in splits.items():
    boxes = data["boxes"]
    n_cajas = [len(b) for b in boxes]
    filas.append(
        {
            "split": name,
            "ventanas": len(boxes),
            "grabaciones": len(recordings[name]),
            "cajas": int(sum(n_cajas)),
            "ventanas_vacias": sum(1 for n in n_cajas if n == 0),
            "cajas_por_ventana": round(sum(n_cajas) / len(n_cajas), 3),
            "max_cajas": max(n_cajas),
            "mel": tuple(data["images"].shape[1:]),
            "dtype": str(data["images"].dtype),
            "GB": round(cache.split_path(name).stat().st_size / 1024**3, 3),
        }
    )

resumen = pd.DataFrame(filas).set_index("split")
resumen.loc["TOTAL"] = [
    resumen["ventanas"].sum(),
    resumen["grabaciones"].sum(),
    resumen["cajas"].sum(),
    resumen["ventanas_vacias"].sum(),
    round(resumen["cajas"].sum() / resumen["ventanas"].sum(), 3),
    resumen["max_cajas"].max(),
    resumen["mel"].iloc[0],
    resumen["dtype"].iloc[0],
    round(resumen["GB"].sum(), 3),
]
resumen

,ventanas,grabaciones,cajas,ventanas_vacias,cajas_por_ventana,max_cajas,mel,dtype,GB
split,,,,,,,,,
train,21883,1324,26935,4514,1.231,11,"(1, 128, 331)",torch.float32,3.464
val,9520,724,10126,2744,1.064,9,"(1, 128, 331)",torch.float32,1.507
test,7972,619,7953,2586,0.998,9,"(1, 128, 331)",torch.float32,1.262
TOTAL,39375,2667,45014,9844,1.143,11,"(1, 128, 331)",torch.float32,6.233


In [10]:
# Reparto en % sobre el total (SPLIT_RATIOS es 0.6 / 0.225 / 0.175 por archivo de audio).
proporciones = resumen.drop(index="TOTAL")[["ventanas", "grabaciones", "cajas"]]
(100 * proporciones / proporciones.sum()).round(2)

,ventanas,grabaciones,cajas
split,,,
train,55.58,49.64,59.84
val,24.18,27.15,22.50
test,20.25,23.21,17.67


### Ninguna grabación puede caer en dos splits

El split es por archivo de audio; si esto falla hay fuga entre train y test.

In [11]:
solapes = {
    f"{a}&{b}": sorted(set(recordings[a]) & set(recordings[b]))
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]
}
for par, comunes in solapes.items():
    print(f"{par}: {len(comunes)} grabaciones en común")

todas = sorted(set().union(*recordings.values()))
print(f"\ngrabaciones en el caché      : {len(todas)}")
print(f"grabaciones del experimento  : {experiment_df['recording'].nunique()}")
faltan = sorted(set(experiment_df["recording"]) - set(todas))
print(f"en el experimento y no en el caché: {len(faltan)}")
for r in faltan:
    print(f"  {r}")

train&val: 0 grabaciones en común
train&test: 0 grabaciones en común
val&test: 0 grabaciones en común

grabaciones en el caché      : 2667
grabaciones del experimento  : 2668
en el experimento y no en el caché: 1
  night_monkey__AA/20240527_175248.wav


### Cajas por clase y split

`cajas` cuenta apariciones en ventanas: una anotación larga cae en varias ventanas
(`clip_len_s` 3 s con `clip_hop_s` 1.5 s), así que siempre supera a `anotaciones`, que son las
anotaciones únicas de las grabaciones asignadas a ese split.

In [12]:
split_de_grabacion = {r: name for name, rs in recordings.items() for r in rs}
experiment_df["split"] = experiment_df["recording"].map(split_de_grabacion)

cajas = pd.DataFrame(
    {
        name: pd.Series(torch.cat(data["labels"]).numpy()).value_counts()
        for name, data in splits.items()
    }
).fillna(0).astype(int)
cajas.index = [labels.names[i] for i in cajas.index]

unicas = (
    experiment_df.pivot_table(index="label", columns="split", values="recording", aggfunc="size")
    .reindex(columns=list(cache.SPLITS))
    .fillna(0)
    .astype(int)
)

por_clase = pd.concat(
    {"cajas": cajas.reindex(index=unicas.index)[list(cache.SPLITS)], "anotaciones": unicas}, axis=1
)
por_clase[("cajas", "total")] = cajas.sum(axis=1)
por_clase[("anotaciones", "total")] = unicas.sum(axis=1)
por_clase.loc["TOTAL"] = por_clase.sum()
por_clase.sort_values(("anotaciones", "total"), ascending=False)

cajas              anotaciones              cajas anotaciones
          train    val  test       train   val  test  total       total
label                                                                  
TOTAL     26935  10126  7953       10296  4075  3193  45014       17564
ac/bc      2887   1082   841        1480   568   439   4810        2487
lw/cs      2668   1000   778        1364   516   402   4446        2282
sm/cc      2359    884   688        1213   461   366   3931        2040
as/bc      2233    837   651        1142   433   334   3721        1909
as/hc      6933   2599  2021         733   384   283  11553        1400
sb/ppc     1267    530   438         650   275   226   2235        1151
aa/gc       922    346   268         474   182   138   1536         794
sb/spc      859    322   251         435   164   130   1432         729
sm/fs       800    298   234         408   158   125   1332         691
pt/dc      1947    730   567         338   169   130   3244         637
lw/trino    726    264   243         373   136   124   1233         633
pt/sqc      465    161   138         239    81    72    764         392
sb/pcc      342    128    99         173    66    51    569         290
lw/ta       311    116    91         159    60    46    518         265
sm/hic      296    111    86         153    57    44    493         254
sm/pc       283    106    83         145    55    42    472         242
ac/chc      281    105    82         140    53    43    468         236
sm/sc       200     76    58         101    39    28    334         168
aa/sc       183     68    53          96    36    27    304         159
ac/sc       188     71    54          96    34    28    313         158
lw/sqc      182     69    52          93    35    26    303         154
ac/gc       191     71    56          79    35    26    318         140
aa/hm       148     54    44          78    28    23    246         129
lw/vc       138     52    40          71    27    21    230         119
sb/sc       126     46    37          63    23    19    209         105

In [13]:
# Anotaciones únicas por grabación y split, por si hay que rastrear una clase rara.
grabaciones_por_clase = (
    experiment_df.pivot_table(index="label", columns="split", values="recording", aggfunc="nunique")
    .reindex(columns=list(cache.SPLITS))
    .fillna(0)
    .astype(int)
)
grabaciones_por_clase["total"] = grabaciones_por_clase.sum(axis=1)
grabaciones_por_clase.sort_values("total", ascending=False)

split,train,val,test,total
label,,,,
as/hc,371,183,152,706
sm/cc,216,131,120,467
pt/dc,141,82,71,294
lw/cs,118,72,58,248
as/bc,95,74,68,237
sb/ppc,103,41,44,188
sb/spc,80,44,42,166
ac/chc,65,42,36,143
ac/bc,61,42,35,138


## 5. Huella para comparar dos copias

`recordings_sha256` resume la lista ordenada de grabaciones de cada split. Si dos copias del
dataset dan la misma huella en los tres splits, el reparto es idéntico; si difiere, las tablas
de arriba dicen dónde.

In [14]:
def huella(items) -> str:
    return hashlib.sha256("\n".join(items).encode()).hexdigest()


huellas = pd.DataFrame(
    [
        {
            "split": name,
            "grabaciones": len(recordings[name]),
            "ventanas": len(splits[name]["boxes"]),
            "cajas": int(sum(len(b) for b in splits[name]["boxes"])),
            "recordings_sha256": huella(recordings[name])[:32],
        }
        for name in cache.SPLITS
    ]
).set_index("split")
huellas

,grabaciones,ventanas,cajas,recordings_sha256
split,,,,
train,1324,21883,26935,9165914cd4ba22c3e31c51d5427a8325
val,724,9520,10126,2450344c98fdcbde063246c12bfe6ed9
test,619,7972,7953,2e403029fea8a88b1c5ee9c9c91ad1d7


In [15]:
reporte = {
    "cleaned": {
        "archivos_txt": len(cleaned_files),
        "grabaciones": int(annotations["recording"].nunique()),
        "anotaciones": int(len(annotations)),
        "pares": int(annotations.groupby(["species", "call_type"]).ngroups),
        "por_carpeta": {
            k: int(v) for k, v in annotations.groupby("folder")["recording"].size().items()
        },
    },
    "experimento": {
        "anotaciones": int(len(experiment_df)),
        "grabaciones": int(experiment_df["recording"].nunique()),
        "clases": list(labels.names),
    },
    "meta": meta,
    "splits": {
        name: {
            "ventanas": len(splits[name]["boxes"]),
            "grabaciones": len(recordings[name]),
            "cajas": int(sum(len(b) for b in splits[name]["boxes"])),
            "ventanas_vacias": int(sum(1 for b in splits[name]["boxes"] if len(b) == 0)),
            "recordings_sha256": huella(recordings[name]),
            "recordings": recordings[name],
        }
        for name in cache.SPLITS
    },
}

destino = ROOT / "notebooks" / "dataset_report.json"
destino.write_text(json.dumps(reporte, indent=2, ensure_ascii=False))
print(f"{destino} ({destino.stat().st_size / 1024:.1f} KB)")
print("Comparalo con la otra copia:  diff <(jq -S . a.json) <(jq -S . b.json)")

/home/fcandia/tesis-primate/notebooks/dataset_report.json (147.4 KB)
Comparalo con la otra copia:  diff <(jq -S . a.json) <(jq -S . b.json)
